# Job Postings: Sample, Methods and Limitations

The reference document behind [What Firms Ask For](./_job_postings_findings_ipynb.ipynb).
Everything the findings page asserts is derived here, and everything it defers
is spelled out here. It reads in the order you would need it to check a number:

1. **The data**: what a record is and where it comes from.
2. **The sample**: which firms are in it, which are not, and why.
3. **Filters**: which postings count, and what each choice costs.
4. **What counts as a mention**: the rules that turn text into features.
5. **How to read the numbers**: four ways of counting, and what each over- and
   under-states.
6. **Results**: the full tables the findings page draws from.
7. **What this cannot support.**

Every table below is built by the project's `doit` pipeline from the committed
corpus. This notebook computes nothing; it loads those tables and shows them.

In [1]:
from jobpostings import report

d = report.load()
m = report.load_methods(d)
report.headline(d)

collection runs: 2026-09-25, 2026-09-27
11,346 postings from 64 firms
9,237 technical postings
1,600 mention a pipeline, ETL/ELT or orchestration


## The data

One record is one job posting, keyed `<firm>:<route>:<native_id>`, where the
route is the applicant-tracking system the firm uses to publish its own openings
(Greenhouse, Workday, Lever and so on). Each record has the title, location,
department and full posting text, and the run on which it was first seen.

Collection is a manual step, run every few months, and it is deliberately not
part of the reproducible build. It touches dozens of firms' careers sites, and
re-running it does not reproduce the past: postings come down once roles are
filled. So the collected corpus is committed to the repository (`data_manual/`),
along with the raw API payloads it was parsed from, and everything from here on
is a pure function of that committed corpus plus one rules file.

A posting seen on several runs is one record with several observations, so the
same tables can later be recomputed per run. The runs so far:

In [2]:
m["collection_runs"]

,Run date,Firms,Postings seen,New,Changed,Newly closed
0,2026-09-25,28,1993,1993,0,0
1,2026-09-27,64,7655,5664,369,2


Until runs are spread over months rather than days, this is one snapshot, and no
trend claim is available.

### Where the language model is, and is not

Finding each firm's endpoint is the one fuzzy step. `src/discover_sources.py`
asks Claude exactly one question per firm: which applicant-tracking system, and
what is the endpoint? That is genuinely hard to automate: one firm's board token
is `wehrtyou`, another's appears only inside JSON embedded in an unrelated page,
a third's differs from the obvious guess by two characters and the obvious guess
returns an empty board.

The answer is then validated by calling the endpoint and requiring real postings
back before anything is written to the manifest. **Claude never reads a posting,
never extracts a feature and never produces a number.** Every count here comes
from the declarative rules described below. The proposer does not have to be
trusted, because the gate confers the trust: a hand-written proposal clears
exactly the same check.

## The sample

The sample frame is `data_manual/manifest.yml`: firms whose careers systems
expose a machine-readable endpoint we have found and validated. That is a real
selection rule with real consequences.

- It is **not a random sample** of quantitative finance. It over-represents
  firms that publish openly and under-represents firms that recruit through
  networks and headhunters.
- Coverage is systematically **worse for banks**, which run fragmented legacy
  applicant-tracking systems.
- Firms differ in board size by **three orders of magnitude**, from a single
  posting to a few thousand. That fact drives the choice of estimator below.

One row per firm that contributed postings, largest board first:

In [3]:
m["firms_collected"]

,Firm,Type,Route,Sampling,Stored scope,Postings,Technical,Pipeline-mentioning
0,JPMorganChase,bank,oracle_recruiting,census,technical_only,2425,2425,501
1,HSBC,bank,eightfold,census,all_postings,1472,176,38
2,Citi,bank,workday,census,technical_only,1023,1023,178
3,Goldman Sachs,bank,oracle_recruiting,census,technical_only,559,559,63
4,Barclays,bank,workday,census,technical_only,421,421,80
5,Morgan Stanley,bank,workday,census,technical_only,400,400,50
6,Deutsche Bank,bank,workday,census,technical_only,361,361,46
7,Jane Street,prop_trading,greenhouse,census,all_postings,230,150,4
8,Bank of America,bank,workday,census,technical_only,220,220,48
9,Point72 (incl. Cubist Systematic Strategies),multistrat,greenhouse,census,all_postings,220,166,23


**Stored scope** matters when comparing the Postings column across firms. Some
adapters (Workday, Oracle) filter to technical titles before storing; the rest
store the whole board. So Postings is not comparable across routes, while
Technical is. The analysis uses technical denominators throughout.

### Firms in the frame with no data

Firms with zero postings are kept rather than dropped, which preserves the
denominator of *attempted* coverage. Their statuses are deliberately
distinguished, because they would otherwise all read as "no data":

- **unresolved**: not found yet. Says nothing about the firm. Discovery is
  non-deterministic: one firm resolved on one run and came back "no
  machine-readable board" on the next with its endpoint working the whole time.
- **unsamplable**: looked at, no route we can use. Mostly this means *we have
  not written the adapter*, not that the firm cannot be reached.
- **broken**: the route works in principle and the firm's own system is failing.

In [4]:
m["firms_without_data"]

,Status,Firm,Type
0,broken,Balyasny Asset Management,multistrat
1,unresolved,Acadian Asset Management,systematic
2,unresolved,BNP Paribas,bank
3,unresolved,Rokos Capital Management,multistrat
4,unresolved,State Street Global Advisors,asset_manager
5,unsamplable,Capula Investment Management,systematic
6,unsamplable,Hudson Bay Capital,multistrat
7,unsamplable,Susquehanna International Group,market_maker
8,unsamplable,UBS,bank


### Per-firm caveats

Recorded in the manifest next to the firm's endpoint and carried into the frame,
so a count and its caveat cannot drift apart.

In [5]:
m["firm_caveats"]

,Firm,Status,Postings,Caveat
0,AQR Capital Management,resolved,51,Postings name essentially no infrastructure products.
1,Arrowstreet Capital,resolved,19,"POST {""appliedFacets"":{},""limit"":20,""offset"":0,""searchText"":""""} returned total=23 and requires offset paging (20 per page); a second site, Campus_Careers, on the same host/tenant holds 4 additional student/intern postings and must be queried separately to get full coverage."
2,Balyasny Asset Management,broken,0,"Workday tenant returns HTTP 401 ""Unable to verify credentials for system account 39$1476"" on the CXS API and serves a ""Workday is currently unavailable"" page for individual roles. Failure is on the firm's side, not a block on us. Retry on each run."
3,Bank of America,resolved,220,"POST to the CXS endpoint returns total=1950 with 20 postings per page (paginate via offset); this Workday site is experienced/lateral hiring only — campus/university roles live on a separate Avature board (bankcampuscareers.tal.net), and no other ghr site name resolved."
4,Barclays,resolved,421,"POST {""appliedFacets"":{},""limit"":20,""offset"":0,""searchText"":""""} returns total=784 and pages correctly at offset=760; Workday caps each response at 20 postings, so full harvest needs ~40 paged requests plus per-job /job/<externalPath> calls for descriptions."
5,Belvedere Trading,resolved,21,"belvederetrading.com/careers links directly to jobs.lever.co/belvederetrading; the v0 postings endpoint returns all 21 live postings in one unpaginated response with full descriptions, so no sampling limitation beyond it showing only currently-published roles."
6,BlackRock,resolved,154,"careers.blackrock.com is a Radancy/TalentBrew front end whose experienced-hire apply links all resolve to this Workday tenant (total 315, paginates cleanly to offset 300); early-career/campus postings are the sampling gap, since those apply through blackrock.tal.net and Beamery flows instead and are not in this Workday site (no other site name on the tenant responded)."
7,Brevan Howard,resolved,12,"POST {""appliedFacets"":{},""limit"":20,""offset"":0,""searchText"":""""} returns total=28 and full pagination (offset=20 yields the remaining 8); brevanhoward.com/careers is a Phenom CareerConnect front-end (tenant BHABHAGB) but the underlying req/apply system is this Workday tenant, so a Phenom-only scrape could in principle surface postings not in this feed."
8,Bridgewater Associates,resolved,15,"Public Greenhouse board carries ~15 roles total, far fewer than the firm hires. Token is only discoverable in the JSON embedded at bridgewater.com/jobboard. Any per-firm rate here rests on a handful of postings."
9,Capital Group,resolved,67,"POST returned total=184 but only 20 per page, so full harvest requires paging offset in increments of 20 (limit caps at 20)."


## Filters, and why

Three decisions sit between "every posting collected" and the pool a share is
computed over.

### 1. Technical titles only

A posting counts as technical if its title matches a short list of stems
(engineer, developer, data, quant, platform, software, scientist, analyst,
research and similar), minus facilities and data-centre roles, which otherwise
produce spurious "airflow" and "data" matches. The pattern is `TECHNICAL_TITLE`
in `src/jobpostings/normalize.py`.

The filter excludes trading roles. That is a choice, so here is what it
discards: postings whose title names a trading role and fails the filter, and
the tool mentions inside them.

In [6]:
m["trading_exclusion"]

,Measure,Value
0,Postings in the corpus,"11,346"
1,Trading-titled postings the filter excludes,157
2,as a share of all postings,1.4%
3,Firms they come from,28
4,Tool mentions inside them,184


In [7]:
m["trading_exclusion_tools"]

,Tool,Postings,Firms
0,Python,77,23
1,SQL (any dialect),34,17
2,R,16,9
3,C++,12,9
4,Data-quality / validation work (generic),10,7
5,Bloomberg,7,6
6,Bash / shell,5,3
7,MATLAB,5,3
8,kdb+ / q,5,2
9,Java,3,3


What is lost is mostly languages and market data, and almost nothing from the
scheduling layer this project is about.

### 2. Which postings form the pool

A share is meaningless without its denominator, so every share names one of
three:

- **all_technical**: every technical posting. Wide; it includes low-latency C++
  and trading roles that would never name an orchestrator, so it *understates*
  prevalence among the people who build pipelines.
- **pipeline_mentioning**: technical postings whose text mentions a pipeline,
  ETL/ELT or orchestration. Much closer to the population the course is about,
  but selected on the outcome's own vocabulary, so it *overstates* prevalence
  relative to engineering hiring generally. **This is the default everywhere
  below and on the findings page.**
- **pipeline_distinct_roles**: the same pool with one row per (firm, title), so
  a role advertised in six offices counts once. The robustness check.

The last two columns show what the choice does to one tool. The share of
*postings* naming Airflow moves several-fold between the wide and narrow pools;
the share of *firms* naming it barely moves. That contrast is the argument for
leading with firms, developed under "How to read the numbers".

In [8]:
m["denominators"]

,Denominator,Postings,Firms,"Airflow, % postings (pooled)","Airflow, % firms naming it",Definition
0,all_technical,9237,63,2.2,57.1,"Every posting whose title looks technical (see TECHNICAL_TITLE in src/jobpostings/normalize.py), excluding facilities and data-centre roles. Wide, and includes many C++/trading roles that would never touch an orchestrator, so it understates tool prevalence among the people who build pipelines."
1,pipeline_mentioning,1600,57,10.2,57.9,"Technical postings whose text mentions a data pipeline, ETL/ELT, or orchestration. Much closer to the population the course is about, but selected on the outcome's vocabulary, so it overstates prevalence relative to all engineering hiring."
2,pipeline_distinct_roles,1349,57,11.0,56.1,"As pipeline_mentioning, but one row per (firm, title), so a role advertised in several offices counts once."


### 3. Distinct roles, not duplicates

Firms post the same role separately for each office. Those are distinct
requisitions, not duplicate records, so the main pools keep them all. The
`pipeline_distinct_roles` pool above is how to check that this does not drive a
result.

## What counts as a mention

Every feature comes from a declarative rule in `src/tool_rules.yml`. The rules
carry context tests because a naive keyword search is wrong in ways that change
conclusions, and they are unit-tested against the exact strings that fooled
earlier versions.

The clearest case is Airflow, which is also a facilities term. At two firms in
this corpus the only keyword matches were data-centre cooling roles, and
counting those would have put a firm that never names an orchestrator near the
top of the Airflow list. The rule as written:

In [9]:
report.code_block(m["rule_airflow"], "yaml")

```yaml
airflow:
  display: Apache Airflow
  layer: orchestrator
  patterns:
  - \bairflow\b
  exclude_if_context:
  - cooling
  - \brack(-|\s)?level\b
  - \bracks?\b
  - CRAH|CRAC|HVAC
  - airflow management
  - liquid cooling
  - containment
  - data cent(er|re)
  - \bCDUs?\b|\bRDHx\b
  note: '"Airflow" is also a facilities term. Jane Street''s and HRT''s only matches were
    data-centre cooling roles ("redundancy topologies, containment, airflow management", "rack-level
    cooling and airflow"). Excluding those is the difference between Jane Street appearing
    to be an Airflow shop and the truth, which is that it never names an orchestrator.'
```

A hit counts only if none of `exclude_if_context` appears within the rule
file's context window (a few hundred characters either side of it). Other traps
the rules guard against, each found by getting it wrong first: `scalable`
matching Scala, `array` matching Ray, "temporal data" matching Temporal.io, a
stack trace matching FINRA TRACE, Monte Carlo simulation matching a
data-observability vendor, and Yarn the JavaScript package manager matching
Hadoop YARN.

### Worked evidence

Counts are only as good as the sentences behind them. These are located with the
same rule that produced the count, at most one per firm, so every excerpt is a
hit that survived the context tests.

**Airflow**

In [10]:
report.quotes_for(m, "airflow")

,Firm,Title,Excerpt
0,Bank of America,Quantitative Finance Analyst,"…ata stores and big data environments. Hands-on expertise with data technologies and computing frameworks including but not limited to, Python, Spark, Airflow, Javascript and SQL. Ability to research new data technologies, architect novel data solutions for business problems and prove design approach throug…"
1,Barclays,AVP - Data Scientist & AI Engineer,"…be expected to work hands-on across on prem & cloud Compliance AI Platform. This requires strong proficiency in data ingestion and processing (Kafka, Airflow, Spark), cloud-based data platforms (Databricks, Snowflake, AWS S3/Azure), and SQL-based data transformation (dbt, PySpark). The candidate must demon…"
2,BlackRock,"Data Engineer – AWS Native Data Platforms, Vice President","…pelines for batch and event‑driven workloads, with a focus on reliability, scalability, and security. Develop and operate data workflows using Apache Airflow for orchestration and Python and SQL for transformation and data quality logic. Implement data transformations and models using modern analytics engi…"
3,Brevan Howard,Front Office Data Engineer,"…ations. Desirable Experience working with market data providers such as Bloomberg, Refinitiv or ICE. Experience with orchestration frameworks such as Airflow, Prefect or Dagster. Experience building internal tools or dashboards using Dash, Streamlit or similar frameworks. Experience with GoldenSource or ot…"
4,Bridgewater Associates,"Research Associate, Equities Data Engineer","…ata-aria-posinset=""5"" data-aria-level=""1""> Familiarity with distributed processing and workflow orchestration (e.g., Spark, Airflow, or equivalents). <li data-leveltext=""""…"
5,Capital Group,Data and AI Engineering Manager,"…ed Qualifications : 10+ years building full-stack data products end-to-end, hands-on across a modern lakehouse stack (Databricks, Unity Catalog, dbt, Airflow/Astronomer); Databricks Apps experience a plus Software engineering depth across multi-tier systems, with hands-on experience delivering information…"


**Control-M**, an enterprise batch scheduler

In [11]:
report.quotes_for(m, "control_m")

,Firm,Title,Excerpt
0,Citi,Applications Support Sr Analyst - Asst. Vice President,"…nce in supporting applications built on Hadoop. Linux 4 - 6 years of experience Database Good SQL experience in any of the RDBMS. Scheduler Autosys / CONTROL-M or other schedulers will be of added advantage. Programming Languages UNIX shell scripting, Python / PERL will be of added advantage. Microsoft Stron…"
1,Deutsche Bank,Engineer,"…g subject areas: Very Good knowledge of the following technologies are needed: Java / Scala, Spark, Hadoop Hive, Workflow orchestrators like Airflow, Control-M, Composer Automation through Python/ Bash / Shell script Cloud offering (GCP preferred) Cloud services - IAAS, PAAS, SAAS Very good Knowledge about t…"
2,Dimensional Fund Advisors,Senior Site Reliability Engineer - Workflow Automation,"…rnetes), DAG authoring best practices, and multi-environment deployments Experience operating enterprise job scheduling platforms (e.g., Automic/UC4, Control-M, etc.) Strong Linux and Windows systems knowledge and comfort working in cloud environments (AWS preferred) Proficiency in Python for automation and…"


**Dagster**, usually listed as an acceptable alternative rather than the core
tool

In [12]:
report.quotes_for(m, "dagster")

,Firm,Title,Excerpt
0,Barclays,Data Engineer - Analyst,"…ood communication skills (both verbal and written) Desirable skills/Preferred Qualifications: • Experience with orchestration tools (Snowflake Tasks, Dagster, Prefect) • Experience in data visualization and data analytics infra using Tableau (or similar BI tool) will be a value add • Past experience with l…"
1,BlackRock,"Senior Data Architect/Data Engineer, Aladdin Engineering - Vice President","…s layer approach. Exposure to orchestration tools (e.g., Directed acyclic graph-based workflow orchestration framework for data and batch processing, Dagster, Prefect) and patterns for dependency management and backfills. Streaming and event-driven data experience (e.g., distributed event streaming and mes…"
2,Brevan Howard,Front Office Data Engineer,"…perience working with market data providers such as Bloomberg, Refinitiv or ICE. Experience with orchestration frameworks such as Airflow, Prefect or Dagster. Experience building internal tools or dashboards using Dash, Streamlit or similar frameworks. Experience with GoldenSource or other enterprise data…"


## How to read the numbers

Because board sizes span three orders of magnitude, "what share of postings name
this tool" is largely a statement about whoever posts the most. So each tool gets
four summaries, and they are reported side by side.

**Notation.** Take one tool and one pool of postings. Firm $i = 1, \dots, F$ has
$n_i$ postings in the pool, of which $x_i$ name the tool, so its own rate is
$\hat p_i = x_i / n_i$.

**% firms naming it** (the extensive margin):

$$
\text{firms\_any} = \frac{1}{F} \sum_{i=1}^{F} \mathbf{1}\{x_i > 0\}.
$$

The most assumption-light. A firm cannot inflate it by posting more, by
reposting one role across offices, or by repeating a stack blurb. It cannot
distinguish one mention from many. The findings page leads with it, alongside
the raw counts $\sum_i x_i$ (Postings) and $\sum_i \mathbf{1}\{x_i > 0\}$ (Firms).

**% postings (pooled)**, total mentions over total postings:

$$
\text{pooled} = \frac{\sum_i x_i}{\sum_i n_i}
  = \sum_i \frac{n_i}{\sum_j n_j}\, \hat p_i .
$$

The second form shows the problem: each firm is weighted by its board size, so a
firm with 2,400 postings outvotes forty firms with 50.

**% postings (firm mean)**, equal weight per firm:

$$
\text{firm\_mean} = \frac{1}{F} \sum_i \hat p_i .
$$

Immune to board size, but a two-posting firm reports 0% or 50% as though either
were a measurement.

**% postings (shrunk)**, the firm mean after pulling each firm's rate toward the
pooled rate in proportion to how little data it has. Assume each firm's true rate
is drawn from a $\text{Beta}(\alpha, \beta)$ distribution and replace
$\hat p_i$ with its posterior mean:

$$
\text{shrunk} = \frac{1}{F} \sum_i \frac{x_i + \alpha}{n_i + \alpha + \beta}
  = \frac{1}{F} \sum_i \Big[\, w_i\, \hat p_i + (1 - w_i)\, \bar p \,\Big],
\qquad w_i = \frac{n_i}{n_i + k},
$$

where $\bar p$ is the pooled rate, $k = \alpha + \beta$ is the prior's weight in
pseudo-postings, $\alpha = \bar p\, k$ and $\beta = (1 - \bar p)\, k$. The prior
is fitted by method of moments: with $v$ the across-firm sample variance of
$\hat p_i$,

$$
k = \frac{\bar p (1 - \bar p)}{v} - 1, \quad \text{clipped to } [2, 200],
$$

falling back to $k = 20$ when $v$ is zero or too large for a beta distribution,
which happens for tools almost nobody names. The second form of the formula is
the interpretation: a firm with $n_i \gg k$ keeps its own rate, and a firm with
$n_i \ll k$ is mostly the pooled rate. `data_manual/index/firm_rates.csv` holds
every $x_i$ and $n_i$, so the shrinkage can be checked by hand.

### The estimators, demonstrated

Two firms: a small one whose every posting names the tool, and one with 1,000
postings that never does. Each row gives the small firm more postings. These are
the production estimators, not a re-implementation.

In [13]:
m["estimator_demo"]

,Small firm's postings (all name it),% firms naming it,% pooled,% firm mean,% shrunk,Prior weight k
0,1,50.0,0.10,50.0,2.43,20.0
1,10,50.0,0.99,50.0,17.01,20.0
2,100,50.0,9.09,50.0,42.51,20.0
3,1000,50.0,50.00,50.0,50.00,20.0


The firm mean always gives the small firm half the vote. Pooled almost never
does. The shrunk estimate earns its way from one to the other as evidence
accumulates. The extensive margin sits at 50% throughout, because it only asks
*whether* a firm names the tool, never how loudly. (With only two firms the
variance estimate is degenerate, so $k$ is at its fallback of 20.)

One property worth stating because it is easy to assume otherwise: the shrunk
estimate does **not** have to lie between the firm mean and the pooled rate.
From the formula, it averages per-firm weights $w_i$ that differ across firms, so
when board size correlates with a firm's own rate it can sit outside that
interval. In this corpus that happens for a sizeable minority of tools, almost
always by a fraction of a percentage point.

### A recorded test: does the weighting hold when one firm gets huge?

Collection was briefly capped at 250 postings per firm as a storage budget. When
the cap was removed (2026-09-28), one firm's contribution grew roughly tenfold
and the corpus grew by half. That was a natural experiment on the estimators.
The table records how far each estimator moved across all tools at the time; it
is a historical record, not recomputed by the pipeline.

| estimator | mean absolute move | max | rows moving >1pp |
|---|---|---|---|
| % postings (firm mean) | 0.078pp | 0.474pp | 0 |
| % postings (shrunk) | 0.410pp | 3.067pp | 9 |
| % postings (pooled) | 0.881pp | 7.319pp | 15 |
| % firms naming it | 1.025pp | 5.263pp | 32 |

Equal weight per firm was essentially immune: not one row moved by a percentage
point. Pooled moved as a volume-weighted statistic must, toward the new firm's
profile (Java +7.3pp, CI/CD +6.9, Kubernetes +5.1, Terraform +4.8), none of which
reflects a change in the market.

The extensive margin moved most. It is immune to *volume inflation of a mention
already counted*, but not to *discovering a mention that truncation had hidden*.
32 tools gained firms and none lost any: a strictly one-directional correction,
so the cap had been biasing the firm count downward. Capping is now opt-in and
unused.

## Results

The full tables the findings page draws from, on the `pipeline_mentioning` pool,
sorted by how many firms name each tool.

### Scheduling and orchestration, every estimator

In [14]:
report.tools_table(d["summary"], layers=report.SCHEDULING_LAYERS)

,Tool,Layer,Postings,Firms,Firms in pool,% firms naming it,% postings (shrunk),% postings (firm mean),% postings (pooled)
0,Apache Airflow,orchestrator,164,33,57,57.9,13.1,13.8,10.2
1,Dagster,orchestrator,23,14,57,24.6,1.8,3.7,1.4
2,Slurm,hpc_scheduler,26,12,57,21.1,2.2,4.1,1.6
3,Argo Workflows / Argo CD,durable_workflow,23,10,57,17.5,1.5,1.6,1.4
4,CA AutoSys,enterprise_scheduler,28,9,57,15.8,1.7,0.9,1.8
5,Prefect,orchestrator,13,9,57,15.8,1.1,3.5,0.8
6,AWS Step Functions,durable_workflow,42,8,57,14.0,2.3,1.3,2.6
7,BMC Control-M,enterprise_scheduler,25,8,57,14.0,1.8,1.6,1.6
8,Azure Data Factory,durable_workflow,10,6,57,10.5,0.6,0.4,0.6
9,Temporal.io,durable_workflow,20,5,57,8.8,1.1,0.4,1.2


### Who names no orchestrator at all

Firms with at least 20 technical postings, none of which names Airflow, Prefect,
Dagster, Luigi, Control-M or AutoSys. This is not random and not a gap in the
data: latency-first proprietary shops describe building their own scheduling
layer instead of adopting a product.

In [15]:
m["no_orchestrator_firms"]

,Firm,Type,Technical postings
0,Jane Street,prop_trading,150
1,Citadel,multistrat,108
2,Jump Trading,prop_trading,94
3,PIMCO,asset_manager,84
4,Hudson River Trading,prop_trading,62
5,Two Sigma,systematic,43
6,The D. E. Shaw Group,systematic,42
7,AQR Capital Management,systematic,36
8,Akuna Capital,market_maker,33
9,Old Mission Capital,market_maker,29


### Every other layer

The full ranking within each layer, so the syllabus can be argued from the data
rather than from the orchestration slice alone.

In [16]:
report.layer_rankings(d["summary"], report.OTHER_LAYERS)

,Layer,Tool,Postings,Firms,% firms naming it,% postings (shrunk)
0,transform,Apache Spark,297,28,49.1,14.6
1,transform,pandas,98,26,45.6,7.2
2,transform,NumPy,69,20,35.1,4.8
3,transform,dbt,50,14,24.6,3.3
4,transform,Ray,31,11,19.3,2.6
5,transform,Dask,21,11,19.3,1.6
6,transform,Polars,20,11,19.3,1.6
7,transform,DuckDB,5,3,5.3,0.4
8,datastore,SQL (any dialect),599,46,80.7,37.5
9,datastore,PostgreSQL,162,32,56.1,12.5


## What this cannot support

- **What firms actually run.** Postings measure hiring intent, not deployed
  infrastructure. A firm can run Airflow for a decade and never name it; a firm
  can name a tool it is only beginning to adopt.
- **Absence as evidence.** Some firms publish their whole stack; others name
  almost nothing (see the caveats table). A zero is ambiguous between "does not
  use it" and "does not say".
- **The industry as a whole.** The frame is firms with a discoverable
  machine-readable board, not a random sample.
- **Per-firm claims** resting on a handful of postings. The shrunk estimator
  exists because those rates are noise.
- **Trends.** Those need several collection runs spread over time. The corpus
  records every observation so the same tables can be recomputed per run once
  enough exist.